In [1]:
import pandas as pd
from src.data import DATA_DIR_INTERIM
from src.config import LLM_NAMES

from src.io import load_qrels_from_path, read_metadata
from topic_gen.evaluate import MetaExperiment
from topic_gen.evaluate.io import load_from_irds
from topic_gen.evaluate.measures_agreement import CohenKappa, AreaUnderReceiver, MeanAverageError
from topic_gen.evaluate.measures_agreement_multiple import LabelDistribution
from topic_gen.evaluate.utils import QrelsTransformer
import ir_datasets

from topic_gen import logger
logger.setLevel("DEBUG")

# Binary

In [2]:
BASE_DIR = DATA_DIR_INTERIM / "dl19" / "qrels-reference"
experiments = load_qrels_from_path(
    BASE_DIR, binarize_qrels=2, drop_relevance_values=999)

Error loading experiment from 2025-12-10_06:31:40: [Errno 2] No such file or directory: '/workspaces/conf26-generating-topics/data/interim/dl19/qrels-reference/2025-12-10_06:31:40/qrels.csv.gz'


In [3]:
baseline = load_from_irds(
    "msmarco-passage/trec-dl-2019/judged", binarize_qrels=2)

In [5]:
meta_exp = MetaExperiment(
    experiments=experiments,
    baseline=baseline,
    measures=[CohenKappa(), AreaUnderReceiver(), MeanAverageError(),LabelDistribution()],
    filter_qrels=True
)

In [6]:
res = meta_exp.evaluate()

In [7]:
metadata = read_metadata(BASE_DIR)

[topic_gen] [WARNING] (io.py:48) Metadata not found for result 2025-12-10_06:31:40, skipping...


In [8]:
df = pd.DataFrame(res)

In [9]:
df

,name,measure,value,missing_topics,missing_qrels
0,2025-12-10_06:31:57,CohenKappa,0.463162,0,0
1,2025-12-05_15:15:10,CohenKappa,0.508248,0,18
2,2025-12-09_07:24:09,CohenKappa,0.533470,0,1
3,2025-12-09_06:32:42,CohenKappa,0.366478,0,0
4,2025-12-07_16:20:49,CohenKappa,0.489429,0,26
5,2025-12-10_06:31:57,AreaUnderReceiver,0.718636,0,0
6,2025-12-05_15:15:10,AreaUnderReceiver,0.742786,0,18
7,2025-12-09_07:24:09,AreaUnderReceiver,0.761177,0,1
8,2025-12-09_06:32:42,AreaUnderReceiver,0.685466,0,0
9,2025-12-07_16:20:49,AreaUnderReceiver,0.737593,0,26


In [ ]:
df = pd.DataFrame(res)
missing = (
    df[["name", "missing_qrels"]].drop_duplicates(
    ).set_index("name").to_dict()["missing_qrels"]
)

df = df.pivot(index="name", columns="measure", values="value").reset_index()

df = df.merge(metadata, left_on="name", right_on="date", how="left")
df["missing"] = df["name"].map(missing)

In [ ]:
df[["model", "prompt", "CohenKappa", "MeanAverageError",
    "AreaUnderReceiver", "missing","label_dist(0)", "label_dist(1)",]].round(2)

,model,prompt,CohenKappa,MeanAverageError,AreaUnderReceiver,missing,label_dist(0),label_dist(1)
0,qwen3-14B-no-think,qrel_zeroshot_bing,0.51,0.21,0.74,0,0.67,0.33
1,gpt-oss-20B,qrel_zeroshot_bing,0.49,0.21,0.74,0,0.70,0.30
2,qwen3-30B-no-think,qrel_zeroshot_bing,0.37,0.32,0.69,0,0.49,0.51
3,gpt-oss-120B-ollama,qrel_zeroshot_bing,0.53,0.19,0.76,0,0.71,0.29
4,deepseek-V3.2,qrel_zeroshot_bing,0.46,0.24,0.72,0,0.63,0.37
5,NaN,NaN,NaN,NaN,NaN,0,0.73,0.27


# Binary Replace Missing

In [ ]:
BASE_DIR = DATA_DIR_INTERIM / "dl19" / "qrels-dl19-reference"
experiments = load_qrels_from_path(
    BASE_DIR, binarize_qrels=2, replace_label_mapping={999: 0})

Error loading experiment from 2025-12-10_06:31:40: [Errno 2] No such file or directory: '/workspaces/conf26-generating-topics/data/interim/dl19/qrels-dl19-reference/2025-12-10_06:31:40/qrels.csv.gz'


In [ ]:
baseline = load_from_irds(
    "msmarco-passage/trec-dl-2019/judged", binarize_qrels=2)

In [ ]:
meta_exp = MetaExperiment(
    experiments=experiments,
    baseline=baseline,
    measures=[CohenKappa(), AreaUnderReceiver(), MeanAverageError(), LabelDistribution()],
    filter_qrels=True
)

In [ ]:
res = meta_exp.evaluate()

In [ ]:
metadata = read_metadata(BASE_DIR)

[topic_gen] [WARNING] (io.py:45) Metadata not found for result 2025-12-10_06:31:40, skipping...


In [ ]:
df = pd.DataFrame(res)
missing = (
    df[["name", "missing_qrels"]].drop_duplicates(
    ).set_index("name").to_dict()["missing_qrels"]
)

df = df.pivot(index="name", columns="measure", values="value").reset_index()

df = df.merge(metadata, left_on="name", right_on="date", how="left")
df["missing"] = df["name"].map(missing)

In [ ]:
df[["name", "model", "prompt", "CohenKappa", "MeanAverageError",
    "AreaUnderReceiver", "missing"]].round(2)

,name,model,prompt,CohenKappa,MeanAverageError,AreaUnderReceiver,missing
0,2025-12-05_15:15:10,qwen3-14B-no-think,qrel_zeroshot_bing,0.51,0.21,0.74,0
1,2025-12-07_16:20:49,gpt-oss-20B,qrel_zeroshot_bing,0.49,0.21,0.74,0
2,2025-12-09_06:32:42,qwen3-30B-no-think,qrel_zeroshot_bing,0.37,0.32,0.69,0
3,2025-12-09_07:24:09,gpt-oss-120B-ollama,qrel_zeroshot_bing,0.53,0.19,0.76,0
4,2025-12-10_06:31:57,deepseek-V3.2,qrel_zeroshot_bing,0.46,0.24,0.72,0
5,msmarco-passage/trec-dl-2019/judged,NaN,NaN,NaN,NaN,NaN,0


## Graded

In [ ]:
BASE_DIR = DATA_DIR_INTERIM / "dl19" / "qrels-dl19-reference"
experiments = load_qrels_from_path(BASE_DIR, replace_label_mapping={999: 0})

Error loading experiment from 2025-12-10_06:31:40: [Errno 2] No such file or directory: '/workspaces/conf26-generating-topics/data/interim/dl19/qrels-dl19-reference/2025-12-10_06:31:40/qrels.csv.gz'


In [ ]:
baseline = load_from_irds("msmarco-passage/trec-dl-2019/judged")

In [ ]:
meta_exp = MetaExperiment(
    experiments=experiments,
    baseline=baseline,
    measures=[CohenKappa(), MeanAverageError(), LabelDistribution()],
    filter_qrels=True
)

In [ ]:
res = meta_exp.evaluate()

In [ ]:
df = pd.DataFrame(res)
missing = (
    df[["name", "missing_qrels"]].drop_duplicates(
    ).set_index("name").to_dict()["missing_qrels"]
)

df = df.pivot(index="name", columns="measure", values="value").reset_index()

df = df.merge(metadata, left_on="name", right_on="date", how="left")
df["missing"] = df["name"].map(missing)

In [ ]:
df[["name", "model", "prompt", "CohenKappa", "MeanAverageError", "missing",
    "label_dist(0)", "label_dist(1)", "label_dist(2)", "label_dist(3)",]].round(2)

,name,model,prompt,CohenKappa,MeanAverageError,missing,label_dist(0),label_dist(1),label_dist(2),label_dist(3)
0,2025-12-05_15:15:10,qwen3-14B-no-think,qrel_zeroshot_bing,0.25,0.68,0,0.26,0.41,0.20,0.13
1,2025-12-07_16:20:49,gpt-oss-20B,qrel_zeroshot_bing,0.24,0.71,0,0.30,0.40,0.12,0.18
2,2025-12-09_06:32:42,qwen3-30B-no-think,qrel_zeroshot_bing,0.09,1.08,0,0.17,0.32,0.14,0.37
3,2025-12-09_07:24:09,gpt-oss-120B-ollama,qrel_zeroshot_bing,0.25,0.68,0,0.30,0.41,0.12,0.17
4,2025-12-10_06:31:57,deepseek-V3.2,qrel_zeroshot_bing,0.29,0.69,0,0.42,0.21,0.16,0.21
5,msmarco-passage/trec-dl-2019/judged,NaN,NaN,NaN,NaN,0,0.56,0.17,0.19,0.08
